# Skill & Role Extraction with Qwen2.5-32B-Instruct

Extracts skills from `resume_text` and `job_description_text`, and extracts the most recent role from resumes.
Adds three new columns: `extracted_resume_skills`, `extracted_job_skills`, and `resume_role`.

## 0. Setup

In [ ]:
import os, json
import pandas as pd
import torch
from tqdm.auto import tqdm
from transformers import AutoTokenizer, AutoModelForCausalLM

In [ ]:
# ── Mount Google Drive for persistent checkpoints ──────────────────────
from google.colab import drive
drive.mount('/content/drive')

# ── Paths ───────────────────────────────────────────────────────────────
# Input data — adjust to wherever your CSVs live
TRAIN_CSV  = "/content/cnamuangtoun:resume-job-description-fit/train.csv"
TEST_CSV   = "/content/cnamuangtoun:resume-job-description-fit/test.csv"

# Output — stored in Drive so everything persists across runtime restarts
OUTPUT_DIR = "/content/drive/MyDrive/skill_extraction"
os.makedirs(OUTPUT_DIR, exist_ok=True)

# Checkpoints so you can resume if interrupted
RESUME_SKILLS_CKPT = os.path.join(OUTPUT_DIR, "resume_skills_checkpoint.json")
JOB_SKILLS_CKPT    = os.path.join(OUTPUT_DIR, "job_skills_checkpoint.json")
RESUME_ROLES_CKPT  = os.path.join(OUTPUT_DIR, "resume_roles_checkpoint.json")

print(f"Output directory: {OUTPUT_DIR}")

## 1. Load data

In [ ]:
train_df = pd.read_csv(TRAIN_CSV)
test_df  = pd.read_csv(TEST_CSV)
all_df   = pd.concat([train_df, test_df], ignore_index=True)

print(f"Total rows: {len(all_df):,}")

# Deduplicate — we only need to extract once per unique text
unique_resumes = all_df[["resume_text"]].drop_duplicates().reset_index(drop=True)
unique_jobs    = all_df[["job_description_text"]].drop_duplicates().reset_index(drop=True)

print(f"Unique resumes: {len(unique_resumes):,}")
print(f"Unique jobs:    {len(unique_jobs):,}")

## 2. Load model

In [ ]:
# Use Qwen2.5-32B-Instruct instead of DeepSeek-R1-Distill.
# R1-Distill always reasons before answering — there is no way to disable it.
# Qwen2.5-32B-Instruct follows instructions directly, same size, same A100 fit.
MODEL_ID = "Qwen/Qwen2.5-32B-Instruct"

llm_tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)

llm_model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    torch_dtype=torch.bfloat16,
    device_map="auto",
)
print(f"Loaded {MODEL_ID}")

## 3. Extraction helpers

In [ ]:
RESUME_PROMPT = """Extract all skills mentioned in the resume below.
Return ONLY a comma-separated list of skills. No explanations, no bullet points, no headers.

Resume:
{text}

Skills:"""

JOB_PROMPT = """Extract all required and preferred skills mentioned in the job description below.
Return ONLY a comma-separated list of skills. No explanations, no bullet points, no headers.

Job Description:
{text}

Skills:"""

ROLE_PROMPT = """Extract the most recent or current job role/title from the resume below.
Return ONLY the job title. No explanations, no bullet points, no headers.
If multiple roles are listed, return only the most recent one.

Resume:
{text}

Job Role:"""


def extract_with_llm(text: str, prompt_template: str, max_new_tokens: int = 256) -> str:
    """Generic LLM extraction function."""
    prompt = prompt_template.format(text=text)
    messages = [{"role": "user", "content": prompt}]
    chat_input = llm_tokenizer.apply_chat_template(
        messages, add_generation_prompt=True, return_tensors="pt", return_dict=True
    )
    input_ids      = chat_input["input_ids"].to(llm_model.device)
    attention_mask = chat_input["attention_mask"].to(llm_model.device)

    with torch.no_grad():
        output_ids = llm_model.generate(
            input_ids=input_ids,
            attention_mask=attention_mask,
            max_new_tokens=max_new_tokens,
            do_sample=False,
        )
    generated = output_ids[0, input_ids.shape[1]:]
    return llm_tokenizer.decode(generated, skip_special_tokens=True).strip()

## 4. Extract resume skills & roles (combined)

In [ ]:
# Load checkpoints if they exist
resume_skills_map = {}
if os.path.exists(RESUME_SKILLS_CKPT):
    with open(RESUME_SKILLS_CKPT, "r") as f:
        resume_skills_map = json.load(f)
    print(f"Resumed skills: {len(resume_skills_map)} resumes already done")

resume_roles_map = {}
if os.path.exists(RESUME_ROLES_CKPT):
    with open(RESUME_ROLES_CKPT, "r") as f:
        resume_roles_map = json.load(f)
    print(f"Resumed roles:  {len(resume_roles_map)} resumes already done")

for i, row in tqdm(unique_resumes.iterrows(), total=len(unique_resumes), desc="Resume skills+roles"):
    text = row["resume_text"]

    if text not in resume_skills_map:
        resume_skills_map[text] = extract_with_llm(text, RESUME_PROMPT)

    if text not in resume_roles_map:
        resume_roles_map[text] = extract_with_llm(text, ROLE_PROMPT, max_new_tokens=64)

    # Checkpoint every 50 resumes
    if (i + 1) % 50 == 0:
        with open(RESUME_SKILLS_CKPT, "w") as f:
            json.dump(resume_skills_map, f, ensure_ascii=False)
        with open(RESUME_ROLES_CKPT, "w") as f:
            json.dump(resume_roles_map, f, ensure_ascii=False)

# Final save
with open(RESUME_SKILLS_CKPT, "w") as f:
    json.dump(resume_skills_map, f, ensure_ascii=False)
with open(RESUME_ROLES_CKPT, "w") as f:
    json.dump(resume_roles_map, f, ensure_ascii=False)

print(f"Done. {len(resume_skills_map)} skills | {len(resume_roles_map)} roles processed.")

## 5. Extract job skills

In [ ]:
# Load checkpoint if exists
job_skills_map = {}
if os.path.exists(JOB_SKILLS_CKPT):
    with open(JOB_SKILLS_CKPT, "r") as f:
        job_skills_map = json.load(f)
    print(f"Resumed: {len(job_skills_map)} jobs already done")

for i, row in tqdm(unique_jobs.iterrows(), total=len(unique_jobs), desc="Job skills"):
    text = row["job_description_text"]
    if text in job_skills_map:
        continue
    job_skills_map[text] = extract_with_llm(text, JOB_PROMPT)

    if (i + 1) % 50 == 0:
        with open(JOB_SKILLS_CKPT, "w") as f:
            json.dump(job_skills_map, f, ensure_ascii=False)

with open(JOB_SKILLS_CKPT, "w") as f:
    json.dump(job_skills_map, f, ensure_ascii=False)
print(f"Done. {len(job_skills_map)} jobs processed.")

## 6. Normalize skills

In [ ]:
from collections import Counter

# ── Manual aliases for known duplicates ─────────────────────────────────
SKILL_ALIASES = {
    "ms excel": "excel",
    "microsoft excel": "excel",
    "ms office": "microsoft office",
    "ms word": "microsoft word",
    "problem solving": "problem-solving",
    "problem-solving skills": "problem-solving",
    "problem solving skills": "problem-solving",
    "communication": "communication skills",
    "written communication": "written communication skills",
    "verbal communication": "verbal communication skills",
    "oral communication": "verbal communication skills",
    "organization": "organizational skills",
    "detail-oriented": "attention to detail",
    "detail oriented": "attention to detail",
    "collaboration": "teamwork",
    "collaborative": "teamwork",
    "analytical": "analytical skills",
    "analysis": "analytical skills",
    "interpersonal": "interpersonal skills",
    "time-management": "time management",
    "c sharp": "c#",
    ".net core": ".net",
    "react.js": "react",
    "reactjs": "react",
    "node.js": "nodejs",
    "vue.js": "vue",
    "vuejs": "vue",
}


def normalize_skill(skill: str) -> str:
    """Lowercase, strip whitespace, apply alias map."""
    s = skill.strip().lower()
    return SKILL_ALIASES.get(s, s)


def normalize_skills_string(raw: str) -> str:
    """Normalize a comma-separated skills string."""
    skills = [normalize_skill(s) for s in raw.split(',') if s.strip()]
    # Deduplicate while preserving order
    seen = set()
    deduped = []
    for s in skills:
        if s not in seen:
            seen.add(s)
            deduped.append(s)
    return ', '.join(deduped)


# ── Apply to both maps (originals preserved in checkpoint files) ───────
resume_skills_norm = {k: normalize_skills_string(v) for k, v in resume_skills_map.items()}
job_skills_norm    = {k: normalize_skills_string(v) for k, v in job_skills_map.items()}

# ── Stats ──────────────────────────────────────────────────────────────
all_raw  = [s.strip().lower() for v in job_skills_map.values() for s in v.split(',') if s.strip()]
all_norm = [s.strip() for v in job_skills_norm.values() for s in v.split(',') if s.strip()]
print(f"Job skills — before: {len(set(all_raw)):,} unique → after: {len(set(all_norm)):,} unique")

all_raw  = [s.strip().lower() for v in resume_skills_map.values() for s in v.split(',') if s.strip()]
all_norm = [s.strip() for v in resume_skills_norm.values() for s in v.split(',') if s.strip()]
print(f"Resume skills — before: {len(set(all_raw)):,} unique → after: {len(set(all_norm)):,} unique")

## 7. Add columns and save

In [ ]:
all_df["extracted_resume_skills"] = all_df["resume_text"].map(resume_skills_norm)
all_df["extracted_job_skills"]    = all_df["job_description_text"].map(job_skills_norm)
all_df["resume_role"]             = all_df["resume_text"].map(resume_roles_map)

# Split back into train/test by original size
n_train = len(train_df)
train_out = all_df.iloc[:n_train]
test_out  = all_df.iloc[n_train:]

train_out.to_csv(os.path.join(OUTPUT_DIR, "train_with_skills.csv"), index=False)
test_out.to_csv(os.path.join(OUTPUT_DIR, "test_with_skills.csv"), index=False)

print(f"Saved train_with_skills.csv ({len(train_out):,} rows)")
print(f"Saved test_with_skills.csv  ({len(test_out):,} rows)")
train_out[["resume_text", "extracted_resume_skills", "resume_role", "job_description_text", "extracted_job_skills", "label"]].head(3)

## 8. Inspect

In [ ]:
import random
sample = train_out.sample(1).iloc[0]

print("LABEL:", sample["label"])
print("\n── RESUME ROLE ──")
print(sample["resume_role"])
print("\n── RESUME SKILLS ──")
print(sample["extracted_resume_skills"])
print("\n── JOB SKILLS ──")
print(sample["extracted_job_skills"])